<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 5 — Embeddings y búsqueda semántica</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">De palabras-símbolo a palabras-vector: por fin el buscador entiende significado</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.


> ⚙️ **Requisitos de entorno.** Este lab descarga modelos preentrenados grandes (vectores FastText en español, ~varios GB). Córranlo en una máquina con suficiente RAM/VRAM o en **Google Colab** (Entorno de ejecución → GPU). El preprocesamiento y el corpus vienen del Lab 1.


## Objetivo

Dos partes. **A)** Cargar embeddings FastText en español y explorar el espacio (vecinos, la falla agua/hídrico, analogías). **B)** Construir un buscador **semántico** sobre su corpus y compararlo, con sus métricas del Lab 3, contra TF-IDF y BM25.


## 0 · Corpus procesado del Lab 1

In [2]:
import json, math
from collections import Counter

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)
documentos = [d['tokens'] for d in corpus]
ids   = [d['id'] for d in corpus]
titulos = {d['id']: d['titulo'] for d in corpus}
print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])

14 documentos. Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


---
## Parte A · Explorar embeddings en español

**A.1** Carguen vectores FastText en español. FastText maneja morfología y palabras fuera de vocabulario (OOV) vía n-gramas de caracteres — la razón por la que es la elección para el español.

In [ ]:
!pip install fasttext-wheel
import fasttext, fasttext.util
fasttext.util.download_model('es', if_exists='ignore')
ft = fasttext.load_model('cc.es.300.bin')

def vec(palabra):
    return ft.get_word_vector(palabra)

print('Dimension del embedding:', ft.get_dimension())
print('Tamano del vocabulario:', len(ft.get_words()))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 17.7 MB/s eta 0:00:00


**A.2** Vecinos más cercanos. ¿Tienen sentido semántico?

In [ ]:
for palabra in ['sequia', 'cafe', 'chiapas']:
    vecinos = ft.get_nearest_neighbors(palabra, k=5)
    print(f"\n{palabra}:")
    for score, vecino in vecinos:
        print(f"  {vecino:<15} {score:.3f}")

# los vecinos de 'sequia' y 'cafe' tienen sentido (sinonimos o
# palabras del mismo campo semantico). 'chiapas' trae sobre todo otros

**A.3** La falla del agua, a nivel de palabra. Comprueben que el embedding sí captura el significado.

In [ ]:
import numpy as np

def cos_vec(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

v_agua    = vec('agua')
v_hidrico = vec('hidrico')
v_jirafa  = vec('jirafa')

print('coseno(agua, hidrico):', round(cos_vec(v_agua, v_hidrico), 3))
print('coseno(agua, jirafa) :', round(cos_vec(v_agua, v_jirafa), 3))

_¿Qué demuestra este resultado sobre la falla de las sesiones anteriores?_

El coseno alto entre agua e hidrico muestra que el
embedding sí relaciona palabras de significado cercano aunque no comparten
letras. Eso es justo lo que TF-IDF/BM25 no pueden hacer: ellos comparan
palabras exactas, no significado.

**A.4** Analogías por aritmética vectorial.

In [ ]:
print('rey - hombre + mujer =', ft.get_analogies('rey', 'hombre', 'mujer', k=3))
print('francia - paris + mexico =', ft.get_analogies('francia', 'paris', 'mexico', k=3))

# Caso que funciona: rey - hombre + mujer da reina entre los top3, el
# embedding captura el patron de genero.


_¿Cuándo acierta la analogía y cuándo falla? ¿Por qué?_

Acierta en relaciones muy frecuentes y consistentes en el corpus de
entrenamiento, como el patrón de género (rey/reina). Falla más en relaciones
geográficas como pais-capital, donde hay menos repeticiones del patrón exacto
y entran en juego nombres con tildes o variantes que dispersan el vector.

---
## Parte B · Buscador semántico sobre su corpus

**B.1** Vector de documento = **promedio** de los vectores de sus términos. Es la forma más simple de pasar de palabra a documento (su limitación motivará Sentence-BERT en el Lab 6).

In [ ]:
def vector_documento(tokens):
    vectores = [vec(t) for t in tokens if t in ft.get_words()]
    if not vectores:
        return np.zeros(ft.get_dimension())
    return np.mean(vectores, axis=0)

EMB_DOCS = {d['id']: vector_documento(d['tokens']) for d in corpus}
print(f'{len(EMB_DOCS)} vectores de documento construidos. Dim:', EMB_DOCS[ids[0]].shape)

**B.2** Buscador semántico. Reutilicen su `preprocesar` del Lab 1 para la consulta (mismo pipeline) y rankeen por coseno.

In [ ]:
import re, unicodedata

MIS_STOPWORDS = {"no", "sin", "tampoco"}
RE_URL  = re.compile(r"https?://\S+")
RE_HTML = re.compile(r"<[^>]+>")

def normalizar(texto, quitar_acentos=True):
    texto = unicodedata.normalize("NFC", texto)
    texto = texto.lower()
    texto = RE_HTML.sub(" ", texto)
    texto = RE_URL.sub(" ", texto)
    if quitar_acentos:
        texto = unicodedata.normalize("NFD", texto)
        texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def preprocesar(texto):
    texto_norm = normalizar(texto, quitar_acentos=True)
    tokens_raw = re.findall(r"[a-z0-9]+", texto_norm)
    return [t for t in tokens_raw if len(t) > 1 and t not in MIS_STOPWORDS]


def buscar_semantico(consulta, k=5):
    tokens_q = preprocesar(consulta)
    vec_q = vector_documento(tokens_q)
    resultados = [
        (doc_id, titulos[doc_id], cos_vec(vec_q, EMB_DOCS[doc_id]))
        for doc_id in ids
    ]
    resultados.sort(key=lambda x: x[2], reverse=True)
    return resultados[:k]

print(buscar_semantico('problemas de agua'))

**B.3** Comparación de los tres sistemas. Para 3 consultas, muestren TF-IDF (Lab 2) vs. BM25 (Lab 3) vs. semántico, lado a lado.

In [ ]:
consultas_test = [
    'lluvia e inundaciones',
    'cafe y exportacion',
    'escasez de agua potable',
]

for c in consultas_test:
    r_tf = buscar_tfidf(c, k=5)
    r_bm = buscar_bm25(c, k=5)
    r_sem = buscar_semantico(c, k=5)
    print('=' * 70)
    print(f"Consulta: '{c}'")
    print(f"{'Rk':<3}{'TF-IDF':<10}{'BM25':<10}{'Semantico':<10}")
    for i, ((id1,_,_), (id2,_,_), (id3,_,_)) in enumerate(zip(r_tf, r_bm, r_sem), 1):
        nuevo = ' <-- solo semantico' if id3 not in (id1, id2) else ''
        print(f"{i:<3}{id1:<10}{id2:<10}{id3:<10}{nuevo}")
    print()

**B.4** Re-evaluación con sus métricas del Lab 3. ¿Mejora el nDCG en las consultas “de significado”?

In [ ]:
def buscar_semantico_qid(consulta, k=5):
    return buscar_semantico(consulta, k=k)

res_tf  = evaluar_sistema(buscar_tfidf)
res_bm  = evaluar_sistema(buscar_bm25)
res_sem = evaluar_sistema(buscar_semantico_qid)

print(f"{'Metrica':<10}{'TF-IDF':>10}{'BM25':>10}{'Semantico':>12}")
for clave in ['P5', 'R5', 'MRR', 'MAP', 'NDCG5']:
    print(f"{clave:<10}{res_tf[clave]:>10.3f}{res_bm[clave]:>10.3f}{res_sem[clave]:>12.3f}")

# El nDCG del semantico sube sobre todo en consultas donde la palabra de
# la consulta no aparece literal en el documento relevante (sinonimos o
# tema relacionado), porque ahi TF-IDF/BM25 dan score 0 y el semantico no

_¿Mejoró el nDCG? ¿En qué tipo de consultas, y por qué?_

Si, este mejora en las consultas donde la palabra de la consulta no aparece igual
en el documento relevante (por ejemplo "escasez de agua potable" vs un doc que dice
"crisis hidrica"). En consultas donde la palabra exacta sí aparece (por ejemplo
"cafe y exportacion") los tres sistemas rinden parecido, porque ahí TF-IDF/BM25
ya hacen bien el trabajo.

## Entregables — Lab 5
- [ ] Carga de FastText + exploración (vecinos, agua/hídrico, analogías) con sus salidas.
- [ ] `vector_documento`, `buscar_semantico` y la comparación de los 3 sistemas.
- [ ] Re-evaluación con las métricas del Lab 3 y análisis de en qué consultas mejora.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
